# Hochladen

In [ ]:
from google.colab import files

# Datei hochladen
model = files.upload()


Saving model.pkl to model.pkl


# Labelig der Daten

In [ ]:
import joblib
import pandas as pd
import requests
from datetime import datetime
import os
import warnings
import matplotlib.pyplot as plt
import shap
from IPython.display import display
from google.colab import files
from pandas import ExcelWriter


# Modell laden
model = joblib.load("model.pkl")

# Testdaten
url = "https://raw.githubusercontent.com/JanisEcker/PraxisProjekt_SAP/refs/heads/main/DataToEvaluate/df100.csv"

# CSV laden
response = requests.get(url)
if response.status_code == 200:
    df_TestData = pd.read_csv(url)
else:
    print(f"Failed to download the file. Status code: {response.status_code}")

# Schwellenwert & Timestamp
threshold = 0.00
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Ordner vorbereiten
base_folder = "criticalTx"
os.makedirs(base_folder, exist_ok=True)

existing_runs = [name for name in os.listdir(base_folder) if os.path.isdir(os.path.join(base_folder, name))]
run_numbers = [int(name.split("_")[0]) for name in existing_runs if name.split("_")[0].isdigit()]
next_run_number = max(run_numbers, default=0) + 1

# Run-Ordnername
run_folder_name = f"{next_run_number:03d}_{timestamp}"

# Zielordner
kritisch_folder = os.path.join(base_folder, run_folder_name)
os.makedirs(kritisch_folder, exist_ok=True)

shap_folder_base = "shap_plots"
shap_folder = os.path.join(shap_folder_base, run_folder_name)
os.makedirs(shap_folder, exist_ok=True)

# Transaktionsnummern prüfen
if 'TxNr' in df_TestData.columns:
    TxNrn = df_TestData['TxNr'].copy()
else:
    TxNrn = pd.Series([f"TX_{i+1:06d}" for i in range(len(df_TestData))])

# Feature-Selektion
feature_columns = [col for col in df_TestData.columns if col not in ["Fraud", "TxNr"]]
X_new = df_TestData[feature_columns]

# Vorhersage berechnen
y_pred = model.predict(X_new)
y_proba = model.predict_proba(X_new)[:, 1]

df_TestData["Prediction"] = y_pred
df_TestData["Fraud_Probability"] = y_proba
df_TestData["Prediction_custom"] = (df_TestData["Fraud_Probability"] > threshold).astype(int)
df_TestData["TxNr"] = TxNrn

# Spalten neu sortieren (TxNr vorne)
cols = ['TxNr'] + [col for col in df_TestData.columns if col != 'TxNr']
df_TestData = df_TestData[cols]

# Kritische Transaktionen extrahieren
kritische_faelle = df_TestData[df_TestData["Prediction_custom"] == 1]

# SHAP-Werte berechnen
explainer = shap.Explainer(model)
shap_values = explainer(X_new)

# SHAP-Werte als DataFrame
shap_df = pd.DataFrame(shap_values.values, columns=X_new.columns)

# Nur SHAP-Werte der kritischen Fälle
shap_df_kritisch = shap_df.loc[kritische_faelle.index].reset_index(drop=True)
kritische_faelle_reset = kritische_faelle.reset_index(drop=True)

# spalten für shap
shap_df_kritisch.columns = [f"SHAP_{col}" for col in shap_df_kritisch.columns]

# shapwerte hinzufügen
kritische_mit_shap = pd.concat([kritische_faelle_reset, shap_df_kritisch], axis=1)

# speichern
excel_path = os.path.join(kritisch_folder, "kritische_transaktionen.xlsx")
with ExcelWriter(excel_path, engine='openpyxl') as writer:
    kritische_mit_shap.to_excel(writer, index=False)


files.download(excel_path)




<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>